In [1]:
!pip install nltk pandas

In [2]:
import re
import json
import nltk
import pandas as pd

from collections import defaultdict
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [3]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

**Part 1 : Text preprocessing**

A.Inspecter le corpus

In [4]:
documents = {
    "D1": "Information retrieval helps users find relevant documents.",
    "D2": "Information Retrieval systems index documents and documents.",
    "D3": "Search engines retrieve relevant web documents.",
    "D4": "Document indexing supports efficient information retrieval.",
    "D5": "Machine learning improves search engine ranking.",
    "D6": "Search systems use indexed documents to retrieve information.",
    "D7": "The search engine retrieves documents; the documents are searchable.",
    "D8": "Indexing documents makes information retrieval more efficient.",
    "D9": "DOCUMENTS contain information about document indexing.",
    "D10": "Retrieval systems organise documents and document collections."
}

**conserver les textes originaux**

In [5]:
for doc_id, text in documents.items():
    print(doc_id, ":", text)

D1 : Information retrieval helps users find relevant documents.
D2 : Information Retrieval systems index documents and documents.
D3 : Search engines retrieve relevant web documents.
D4 : Document indexing supports efficient information retrieval.
D5 : Machine learning improves search engine ranking.
D6 : Search systems use indexed documents to retrieve information.
D7 : The search engine retrieves documents; the documents are searchable.
D8 : Indexing documents makes information retrieval more efficient.
D9 : DOCUMENTS contain information about document indexing.
D10 : Retrieval systems organise documents and document collections.


**B. Version A : Lowercase + punctuation removal**

In [6]:
def preprocess_document(text):

    # 1. Convertir le texte en minuscules
    text = text.lower()

    # 2. Supprimer la ponctuation
    text = re.sub(r'[^\w\s]', '', text)

    # 3. Diviser le texte en tokens
    tokens = text.split()

    return tokens

In [7]:
tokens_A = {}

for doc_id, text in documents.items():
    tokens_A[doc_id] = preprocess_document(text)

for doc_id, tokens in tokens_A.items():
    print(doc_id, ":", tokens)

D1 : ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents']
D2 : ['information', 'retrieval', 'systems', 'index', 'documents', 'and', 'documents']
D3 : ['search', 'engines', 'retrieve', 'relevant', 'web', 'documents']
D4 : ['document', 'indexing', 'supports', 'efficient', 'information', 'retrieval']
D5 : ['machine', 'learning', 'improves', 'search', 'engine', 'ranking']
D6 : ['search', 'systems', 'use', 'indexed', 'documents', 'to', 'retrieve', 'information']
D7 : ['the', 'search', 'engine', 'retrieves', 'documents', 'the', 'documents', 'are', 'searchable']
D8 : ['indexing', 'documents', 'makes', 'information', 'retrieval', 'more', 'efficient']
D9 : ['documents', 'contain', 'information', 'about', 'document', 'indexing']
D10 : ['retrieval', 'systems', 'organise', 'documents', 'and', 'document', 'collections']


C. Version B : Stop-word removal

Récupérer la liste des stop words

In [8]:
stop_words = set(stopwords.words('english'))

print(stop_words)

{'he', 'she', 'won', "we've", 'having', 're', "should've", 'i', "isn't", 'had', 'shouldn', 'yourselves', 'mustn', 'those', 'there', 'has', 'hadn', "they're", 'into', 'my', 'theirs', 'very', 'isn', "she'll", 'when', "i'll", 'most', "they'd", 'any', 'once', 'after', 'are', 'am', 'because', 'ourselves', 'for', 'and', 'not', 'weren', 'below', 'of', "you're", 'wasn', "didn't", "you'll", 'until', 'its', "wouldn't", "don't", 'own', 'all', 'is', 'above', "that'll", 'or', 'himself', 'our', "it's", 'here', "needn't", 'against', 'through', 'him', 'you', 'same', 'under', 'be', 'more', "i'd", 'few', 'doesn', "she'd", "he's", 'ours', 'me', "he'd", 'by', "we're", "won't", 'during', 'hasn', 'such', "shouldn't", "we'd", 'will', 'll', 'myself', 'if', "shan't", 'your', 'o', 'before', 'haven', 'couldn', 'then', 'what', 'where', 'why', 'being', 'his', 'been', "mustn't", 'as', "hasn't", 'yourself', "you'd", 'aren', 'on', 'y', 'both', 'do', 'too', "we'll", 'up', 'their', 'at', 'to', 'about', "weren't", 'now'

**Créer une fonction pour supprimer les stop words**

In [9]:
def remove_stopwords(tokens):

    tokens_without_stopwords = [
        word for word in tokens
        if word not in stop_words
    ]

    return tokens_without_stopwords

**Appliquaction de cette fonction à tous les documents**

In [10]:
tokens_B = {}

for doc_id, tokens in tokens_A.items():
    tokens_B[doc_id] = remove_stopwords(tokens)

for doc_id, tokens in tokens_B.items():
    print(doc_id, ":", tokens)

D1 : ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents']
D2 : ['information', 'retrieval', 'systems', 'index', 'documents', 'documents']
D3 : ['search', 'engines', 'retrieve', 'relevant', 'web', 'documents']
D4 : ['document', 'indexing', 'supports', 'efficient', 'information', 'retrieval']
D5 : ['machine', 'learning', 'improves', 'search', 'engine', 'ranking']
D6 : ['search', 'systems', 'use', 'indexed', 'documents', 'retrieve', 'information']
D7 : ['search', 'engine', 'retrieves', 'documents', 'documents', 'searchable']
D8 : ['indexing', 'documents', 'makes', 'information', 'retrieval', 'efficient']
D9 : ['documents', 'contain', 'information', 'document', 'indexing']
D10 : ['retrieval', 'systems', 'organise', 'documents', 'document', 'collections']


**D. Version C : Stemming avec PorterStemmer**

In [11]:
#Initialiser PorterStemmer
stemmer = PorterStemmer()

In [12]:
#Créer la fonction de stemming
def apply_stemming(tokens):

    stemmed_tokens = [
        stemmer.stem(word)
        for word in tokens
    ]

    return stemmed_tokens

 Appliquer le stemming aux documents de la version B

In [13]:
tokens_C = {}

for doc_id, tokens in tokens_B.items():
    tokens_C[doc_id] = apply_stemming(tokens)

for doc_id, tokens in tokens_C.items():
    print(doc_id, ":", tokens)

D1 : ['inform', 'retriev', 'help', 'user', 'find', 'relev', 'document']
D2 : ['inform', 'retriev', 'system', 'index', 'document', 'document']
D3 : ['search', 'engin', 'retriev', 'relev', 'web', 'document']
D4 : ['document', 'index', 'support', 'effici', 'inform', 'retriev']
D5 : ['machin', 'learn', 'improv', 'search', 'engin', 'rank']
D6 : ['search', 'system', 'use', 'index', 'document', 'retriev', 'inform']
D7 : ['search', 'engin', 'retriev', 'document', 'document', 'searchabl']
D8 : ['index', 'document', 'make', 'inform', 'retriev', 'effici']
D9 : ['document', 'contain', 'inform', 'document', 'index']
D10 : ['retriev', 'system', 'organis', 'document', 'document', 'collect']


**E.Comparer les trois versions**

In [14]:
for doc_id in documents:

    print("\n", doc_id)

    print("Version A :", tokens_A[doc_id])

    print("Version B :", tokens_B[doc_id])

    print("Version C :", tokens_C[doc_id])


 D1
Version A : ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents']
Version B : ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents']
Version C : ['inform', 'retriev', 'help', 'user', 'find', 'relev', 'document']

 D2
Version A : ['information', 'retrieval', 'systems', 'index', 'documents', 'and', 'documents']
Version B : ['information', 'retrieval', 'systems', 'index', 'documents', 'documents']
Version C : ['inform', 'retriev', 'system', 'index', 'document', 'document']

 D3
Version A : ['search', 'engines', 'retrieve', 'relevant', 'web', 'documents']
Version B : ['search', 'engines', 'retrieve', 'relevant', 'web', 'documents']
Version C : ['search', 'engin', 'retriev', 'relev', 'web', 'document']

 D4
Version A : ['document', 'indexing', 'supports', 'efficient', 'information', 'retrieval']
Version B : ['document', 'indexing', 'supports', 'efficient', 'information', 'retrieval']
Version C : ['document', 'index', 'support', 'effici